# Silver Layer — Delta Lake

**Today's task:** Enhance the batch pipeline by storing preprocessed S&P 500 data as **Delta Lake tables** in the silver layer (instead of Parquet on S3).

| Layer | Path | Format |
|-------|------|--------|
| Bronze (raw) | `s3://stocks-bronze-layer/raw/batch/` | CSV |
| Silver (cleaned) | `s3://stocks-silver-layer/processed/batch/` | **Delta** |

Run this notebook on a Databricks cluster with access to both S3 buckets.

## 1. Config

In [ ]:
BRONZE_PATH = "s3://stocks-bronze-layer/raw/batch/SP500_Historical_Data.csv"
SILVER_PATH = "s3://stocks-silver-layer/processed/batch/"
SILVER_TABLE = "silver.sp500_historical"

print("Bronze:", BRONZE_PATH)
print("Silver:", SILVER_PATH)
print("Table :", SILVER_TABLE)

## 2. Verify bronze source

In [ ]:
display(dbutils.fs.ls("s3://stocks-bronze-layer/raw/batch/"))

## 3. Read bronze CSV

In [ ]:
df = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv(BRONZE_PATH)
)

print("Rows loaded:", df.count())
df.printSchema()
display(df.limit(5))

## 4. Data quality checks (bronze)

In [ ]:
from pyspark.sql.functions import col, sum as spark_sum

# Nulls
null_counts = df.select([
    spark_sum(col(c).isNull().cast("int")).alias(c)
    for c in df.columns
])
display(null_counts)

# Duplicate ticker-date keys
duplicate_count = (
    df.groupBy("Ticker", "Date")
      .count()
      .filter(col("count") > 1)
      .count()
)
print("Duplicate Ticker-Date combinations:", duplicate_count)

# Invalid OHLC / volume rows
invalid_prices = df.filter(
    (col("Open") <= 0) |
    (col("High") <= 0) |
    (col("Low") <= 0) |
    (col("Close") <= 0) |
    (col("Adj Close") <= 0) |
    (col("Volume") < 0) |
    (col("High") < col("Low"))
)
print("Invalid price records:", invalid_prices.count())

## 5. Preprocess for Silver

Clean, type, deduplicate, and add partition / lineage columns.

In [ ]:
from pyspark.sql.functions import to_date, year, month, current_timestamp, col

# Typed date + snake_case columns
df_silver = (
    df
    .withColumn("Date", to_date(col("Date"), "yyyy-MM-dd"))
    .toDF(
        "ticker",
        "date",
        "open",
        "high",
        "low",
        "close",
        "adj_close",
        "volume",
    )
)

print("Invalid dates after cast:", df_silver.filter(col("date").isNull()).count())

# Keep only valid OHLCV rows
df_silver = df_silver.filter(
    (col("ticker").isNotNull()) &
    (col("date").isNotNull()) &
    (col("open") > 0) &
    (col("high") > 0) &
    (col("low") > 0) &
    (col("close") > 0) &
    (col("adj_close") > 0) &
    (col("volume") >= 0) &
    (col("high") >= col("low"))
)

# One row per ticker-date
df_silver = df_silver.dropDuplicates(["ticker", "date"])

# Partition helpers + lineage
df_silver = (
    df_silver
    .withColumn("year", year(col("date")))
    .withColumn("month", month(col("date")))
    .withColumn("processed_at", current_timestamp())
)

print("Preprocessed row count:", df_silver.count())
df_silver.printSchema()
display(df_silver.limit(5))

## 6. Cutover check (Parquet → Delta)

If `SILVER_PATH` already has Parquet (or other) files but **no** `_delta_log/`, Delta cannot overwrite it. Clear that path once, then write Delta.

In [ ]:
def is_delta_path(path: str) -> bool:
    try:
        return any(f.name.rstrip("/").endswith("_delta_log") for f in dbutils.fs.ls(path))
    except Exception:
        return False  # path does not exist yet

if is_delta_path(SILVER_PATH):
    print("Existing Delta table found at silver path. Overwrite is safe.")
else:
    # Old Parquet / empty / mixed path — remove so Delta can create _delta_log
    try:
        dbutils.fs.ls(SILVER_PATH)
        print(f"Non-Delta files found at {SILVER_PATH}. Clearing for Delta cutover...")
        dbutils.fs.rm(SILVER_PATH, True)
    except Exception:
        print("Silver path is empty or missing. Delta write will create it.")

print("Ready to write Silver as Delta Lake.")

## 7. Write Silver as a Delta Lake table

Replaces previous Parquet writes. Data is partitioned by `year` / `month` and registered as an external Delta table for SQL.

In [ ]:
(
    df_silver.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .partitionBy("year", "month")
    .save(SILVER_PATH)
)

# Register external Delta table for SQL (replace if an old definition exists)
spark.sql("CREATE SCHEMA IF NOT EXISTS silver")
spark.sql(f"DROP TABLE IF EXISTS {SILVER_TABLE}")
spark.sql(f"""
CREATE TABLE {SILVER_TABLE}
USING DELTA
LOCATION '{SILVER_PATH}'
""")

print("Preprocessed data written to Silver Delta table.")
print("Path :", SILVER_PATH)
print("Table:", SILVER_TABLE)

## 8. Verify Delta files & table

In [ ]:
# Confirm _delta_log exists (proves Delta, not plain Parquet)
display(dbutils.fs.ls(SILVER_PATH))
display(dbutils.fs.ls(f"{SILVER_PATH}_delta_log/"))

In [ ]:
# Read via path
silver_df = spark.read.format("delta").load(SILVER_PATH)

# Equivalent via registered table:
# silver_df = spark.table(SILVER_TABLE)

print("Silver Delta row count:", silver_df.count())
silver_df.printSchema()
display(silver_df.limit(5))

In [ ]:
# Delta transaction history + table details
display(spark.sql(f"DESCRIBE HISTORY delta.`{SILVER_PATH}`"))
display(spark.sql(f"DESCRIBE DETAIL delta.`{SILVER_PATH}`"))
display(spark.sql(f"SHOW CREATE TABLE {SILVER_TABLE}"))

## 9. Sample SQL queries on the Silver Delta table

In [ ]:
display(spark.sql(f"""
SELECT ticker, date, open, high, low, close, volume, year, month
FROM {SILVER_TABLE}
WHERE ticker = 'AAPL'
ORDER BY date DESC
LIMIT 10
"""))

display(spark.sql(f"""
SELECT year, month, COUNT(*) AS row_count
FROM {SILVER_TABLE}
GROUP BY year, month
ORDER BY year, month
"""))